# 🚀 Sistema RAG con Self-Correction (Nivel Avanzado)

## 🎯 ¿Qué construiremos?

Un **Sistema RAG Agéntico** (Retrieval-Augmented Generation) con capacidad de **auto-corrección**.

### 🧠 ¿Qué hace diferente a este RAG?

**RAG Básico (Junior):**
```
Pregunta → Buscar docs → Generar respuesta → Fin
```

**RAG con Self-Correction (Senior):**
```
Pregunta → Buscar docs → Generar respuesta → ¿Es buena?
                ↑                                  |
                |                                  |
                └────── Buscar mejor ──────── No ─┘
                                               |
                                              Sí → Respuesta final
```

### ✨ Características Avanzadas:

1. **Evaluación de calidad**: El sistema evalúa su propia respuesta
2. **Self-correction**: Si no está seguro, busca más información
3. **Múltiples intentos**: Puede iterar hasta conseguir una buena respuesta
4. **Relevance checking**: Verifica si los documentos son relevantes
5. **Confidence scoring**: Mide qué tan confiado está en su respuesta

---

## 🎓 Conceptos que aprenderás:

- ✅ RAG avanzado con múltiples iteraciones
- ✅ Embeddings y búsqueda semántica
- ✅ Vector stores (FAISS)
- ✅ Evaluación automática de calidad
- ✅ Flujos de decisión complejos
- ✅ Ciclos de retroalimentación inteligentes

---

## 🏗️ Arquitectura del Sistema:

```
┌─────────────────────────────────────────────────┐
│              USUARIO                            │
│         "¿Qué es LangGraph?"                    │
└─────────────────┬───────────────────────────────┘
                  ↓
         ┌────────────────┐
         │  RETRIEVE      │  ← Busca en vector store
         └────────┬───────┘
                  ↓
         ┌────────────────┐
         │  CHECK         │  ← ¿Docs relevantes?
         │  RELEVANCE     │
         └────┬───────────┘
              ↓
       ┌──────┴──────┐
       │             │
      Sí            No → Web Search (fallback)
       │             │
       ↓             ↓
  ┌────────────────────┐
  │   GENERATE         │  ← Crea respuesta
  └─────────┬──────────┘
            ↓
  ┌────────────────────┐
  │   EVALUATE         │  ← ¿Respuesta buena?
  └─────────┬──────────┘
            ↓
      ┌─────┴─────┐
      │           │
  Alta conf.   Baja conf.
      │           │
      ↓           ↓
    END      RETRIEVE (nuevo intento)
```

---

## 📦 Paso 1: Instalación de Bibliotecas

Necesitamos bibliotecas adicionales para RAG:

- **`faiss-cpu`**: Vector database para búsqueda semántica
- **`langchain-openai`**: Embeddings y LLM de OpenAI
- **`langgraph`**: Framework para el flujo
- **`tiktoken`**: Para contar tokens

In [ ]:
# Instalación de todas las dependencias
!pip install -q --upgrade \
    langgraph \
    langchain \
    langchain-openai \
    langchain-community \
    faiss-cpu \
    tiktoken

print("✅ Todas las bibliotecas instaladas!")

---

## 📥 Paso 2: Imports

In [ ]:
# Imports principales
import os
from typing import TypedDict, Annotated, List
from typing_extensions import TypedDict

# LangChain
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter  # ← Cambio aquí
from langchain_core.documents import Document

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages

print("✅ Imports completados!")

---

## 🔑 Paso 3: Configurar API Key

---

## 📚 Paso 4: Crear la Base de Conocimiento

### 📝 Explicación

Primero necesitamos **documentos** para nuestro sistema RAG. Vamos a crear una pequeña base de conocimiento sobre LangGraph y IA.

En un sistema real, estos documentos vendrían de:
- PDFs
- Bases de datos
- APIs
- Sitios web
- Etc.

In [ ]:
# Crear documentos de ejemplo sobre LangGraph y temas relacionados
documents = [
    Document(
        page_content="LangGraph es un framework de Python para construir aplicaciones con grafos stateful usando LLMs. "
        "Permite crear flujos de trabajo complejos con nodos y edges, soportando tanto DAGs como grafos cíclicos. "
        "Es especialmente útil para crear agentes que pueden tomar decisiones y mantener estado entre pasos.",
        metadata={"source": "langgraph_intro", "topic": "langgraph"}
    ),
    Document(
        page_content="Los agentes de IA son sistemas que pueden razonar, planificar y actuar de forma autónoma. "
        "Usan LLMs como motor de razonamiento y pueden acceder a herramientas externas. "
        "Los buenos agentes tienen memoria, pueden corregirse a sí mismos, y aprenden de sus errores.",
        metadata={"source": "ai_agents", "topic": "agents"}
    ),
    Document(
        page_content="RAG (Retrieval-Augmented Generation) es una técnica que combina búsqueda de información con generación de texto. "
        "Primero recupera documentos relevantes de una base de conocimiento, luego usa esos documentos como contexto "
        "para que el LLM genere una respuesta más precisa y fundamentada.",
        metadata={"source": "rag_basics", "topic": "rag"}
    ),
    Document(
        page_content="Los embeddings son representaciones vectoriales de texto que capturan significado semántico. "
        "Permiten medir similaridad entre textos usando distancia coseno. "
        "Son fundamentales para sistemas de búsqueda semántica y RAG.",
        metadata={"source": "embeddings_info", "topic": "embeddings"}
    ),
    Document(
        page_content="FAISS (Facebook AI Similarity Search) es una biblioteca para búsqueda eficiente de vectores similares. "
        "Permite indexar millones de vectores y hacer búsquedas en milisegundos. "
        "Es muy usada en sistemas RAG para encontrar documentos relevantes rápidamente.",
        metadata={"source": "faiss_info", "topic": "vector_db"}
    ),
    Document(
        page_content="LangChain es un framework para desarrollar aplicaciones con LLMs. "
        "Proporciona componentes modulares como prompts, chains, agents, y memory. "
        "Facilita la integración con múltiples LLMs y herramientas externas.",
        metadata={"source": "langchain_intro", "topic": "langchain"}
    ),
    Document(
        page_content="Los prompts son instrucciones que se le dan a un LLM para obtener respuestas específicas. "
        "Un buen prompt engineering incluye contexto claro, ejemplos, y formato estructurado. "
        "La calidad del prompt afecta directamente la calidad de la respuesta.",
        metadata={"source": "prompt_engineering", "topic": "prompts"}
    ),
    Document(
        page_content="Los vectorstores almacenan embeddings y permiten búsqueda semántica eficiente. "
        "Ejemplos incluyen FAISS, Pinecone, Chroma, y Weaviate. "
        "Son esenciales para implementar RAG a escala.",
        metadata={"source": "vectorstores", "topic": "vector_db"}
    ),
]

print(f"✅ Creados {len(documents)} documentos para la base de conocimiento")
print("\n📚 Temas cubiertos:")
topics = set([doc.metadata['topic'] for doc in documents])
for topic in topics:
    print(f"   • {topic}")

---

## 🗄️ Paso 5: Crear el Vector Store

### 📝 Explicación

Ahora convertimos los documentos en **embeddings** (vectores) y los almacenamos en un **vector store** (FAISS).

**Proceso:**
1. Cada documento se convierte en un vector numérico
2. Los vectores capturan el significado semántico del texto
3. FAISS indexa estos vectores para búsqueda rápida
4. Cuando hacemos una pregunta, se buscan los vectores más similares

In [ ]:
# Crear embeddings model
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Crear el vector store con FAISS
vectorstore = FAISS.from_documents(documents, embeddings)

# Crear el retriever (k=3 significa que recupera los 3 documentos más relevantes)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✅ Vector store creado con FAISS!")
print(f"📊 Vectores indexados: {len(documents)}")
print("🔍 Retriever configurado para buscar top-3 documentos más relevantes")

---

## 🧪 Paso 6: Probar el Retriever

Vamos a hacer una prueba rápida para ver si el retriever funciona correctamente.

In [ ]:
# Prueba del retriever
test_query = "¿Qué es LangGraph?"

retrieved_docs = retriever.invoke(test_query)

print(f"🔍 Pregunta de prueba: '{test_query}'")
print(f"\n📚 Documentos recuperados: {len(retrieved_docs)}\n")
print("=" * 70)

for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n[{i}] Fuente: {doc.metadata['source']}")
    print(f"Contenido: {doc.page_content[:150]}...")
    print("-" * 70)

print("\n✅ Retriever funcionando correctamente!")

---

## 🤖 Paso 7: Configurar el LLM

In [ ]:
# Configurar el modelo de lenguaje
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ LLM configurado!")
print("📌 Modelo: gpt-4o-mini")
print("🌡️ Temperature: 0 (respuestas determinísticas)")

---

## 🏗️ Paso 8: Definir el Estado del Grafo

### 📝 Explicación

El estado mantendrá toda la información necesaria a través del flujo:

- **`question`**: La pregunta del usuario
- **`documents`**: Documentos recuperados
- **`answer`**: La respuesta generada
- **`relevance_score`**: ¿Son relevantes los docs? (0-10)
- **`confidence_score`**: ¿Qué tan confiado está? (0-10)
- **`retrieval_attempts`**: Cuántas veces ha buscado
- **`messages`**: Historial de mensajes

In [ ]:
# Definir el estado del grafo RAG
class RAGState(TypedDict):
    question: str                    # Pregunta del usuario
    documents: List[Document]        # Documentos recuperados
    answer: str                      # Respuesta generada
    relevance_score: int            # Score de relevancia (0-10)
    confidence_score: int           # Score de confianza (0-10)
    retrieval_attempts: int         # Número de intentos de recuperación
    messages: Annotated[list, add_messages]  # Historial de mensajes

print("✅ Estado del grafo definido!")
print("\n📊 Campos del estado:")
print("   • question: Pregunta del usuario")
print("   • documents: Docs recuperados")
print("   • answer: Respuesta generada")
print("   • relevance_score: ¿Docs relevantes? (0-10)")
print("   • confidence_score: ¿Confianza en respuesta? (0-10)")
print("   • retrieval_attempts: Número de búsquedas")
print("   • messages: Historial completo")

---

## 🎯 Paso 9: Definir los Nodos del Grafo

Vamos a crear los 4 nodos principales:

1. **RETRIEVE**: Busca documentos relevantes
2. **CHECK_RELEVANCE**: Evalúa si los docs son relevantes
3. **GENERATE**: Crea la respuesta
4. **EVALUATE**: Evalúa la calidad de la respuesta

### 🔍 Nodo 1: RETRIEVE

Busca documentos en el vector store.

In [ ]:
def retrieve_documents(state: RAGState) -> RAGState:
    """
    Recupera documentos relevantes del vector store.
    """
    question = state["question"]
    attempts = state.get("retrieval_attempts", 0)
    
    print(f"\n🔍 RETRIEVE (Intento #{attempts + 1})")
    print(f"   Pregunta: {question}")
    
    # Recuperar documentos
    docs = retriever.invoke(question)
    
    print(f"   ✅ Recuperados {len(docs)} documentos")
    
    return {
        **state,
        "documents": docs,
        "retrieval_attempts": attempts + 1
    }

print("✅ Nodo RETRIEVE definido!")

### ✅ Nodo 2: CHECK_RELEVANCE

Evalúa si los documentos recuperados son relevantes para la pregunta.

In [ ]:
def check_relevance(state: RAGState) -> RAGState:
    """
    Evalúa la relevancia de los documentos recuperados.
    """
    question = state["question"]
    docs = state["documents"]
    
    print("\n✅ CHECK_RELEVANCE")
    print("   Evaluando relevancia de documentos...")
    
    # Crear prompt para evaluar relevancia
    eval_prompt = f"""
Evalúa si los siguientes documentos son relevantes para responder la pregunta.

Pregunta: {question}

Documentos:
{chr(10).join([f"{i+1}. {doc.page_content[:200]}..." for i, doc in enumerate(docs)])}

Responde SOLO con un número del 0 al 10, donde:
- 0-3: No relevantes (necesitas buscar otra cosa)
- 4-6: Parcialmente relevantes (podrían servir)
- 7-10: Muy relevantes (perfectos para responder)

Score de relevancia (solo el número):"""
    
    response = llm.invoke(eval_prompt)
    
    # Extraer el score
    try:
        relevance_score = int(response.content.strip())
    except:
        relevance_score = 5  # Default si no puede parsear
    
    print(f"   📊 Score de relevancia: {relevance_score}/10")
    
    return {
        **state,
        "relevance_score": relevance_score
    }

print("✅ Nodo CHECK_RELEVANCE definido!")

### 📝 Nodo 3: GENERATE

Genera la respuesta usando los documentos recuperados como contexto.

In [ ]:
def generate_answer(state: RAGState) -> RAGState:
    """
    Genera una respuesta basada en los documentos recuperados.
    """
    question = state["question"]
    docs = state["documents"]
    
    print("\n📝 GENERATE")
    print("   Generando respuesta...")
    
    # Crear contexto con los documentos
    context = "\n\n".join([doc.page_content for doc in docs])
    
    # Crear prompt para generar respuesta
    gen_prompt = f"""
Eres un asistente experto. Responde la pregunta usando SOLO la información del contexto proporcionado.

Contexto:
{context}

Pregunta: {question}

Instrucciones:
- Responde de forma clara y concisa
- Usa SOLO información del contexto
- Si el contexto no tiene suficiente información, di "No tengo suficiente información"
- Sé específico y preciso

Respuesta:"""
    
    response = llm.invoke(gen_prompt)
    answer = response.content
    
    print(f"   ✅ Respuesta generada ({len(answer)} caracteres)")
    
    return {
        **state,
        "answer": answer
    }

print("✅ Nodo GENERATE definido!")

### 🎯 Nodo 4: EVALUATE

Evalúa la calidad y confianza de la respuesta generada.

In [ ]:
def evaluate_answer(state: RAGState) -> RAGState:
    """
    Evalúa la calidad y confianza de la respuesta.
    """
    question = state["question"]
    answer = state["answer"]
    docs = state["documents"]
    
    print("\n🎯 EVALUATE")
    print("   Evaluando calidad de la respuesta...")
    
    # Crear prompt para evaluar confianza
    eval_prompt = f"""
Evalúa la calidad de esta respuesta considerando:
- ¿Responde completamente la pregunta?
- ¿Es precisa y específica?
- ¿Está bien fundamentada en el contexto?

Pregunta: {question}

Respuesta: {answer}

Responde SOLO con un número del 0 al 10, donde:
- 0-3: Respuesta pobre (necesita más información)
- 4-6: Respuesta aceptable (podría mejorar)
- 7-10: Respuesta excelente (muy confiable)

Score de confianza (solo el número):"""
    
    response = llm.invoke(eval_prompt)
    
    # Extraer el score
    try:
        confidence_score = int(response.content.strip())
    except:
        confidence_score = 5  # Default si no puede parsear
    
    print(f"   📊 Score de confianza: {confidence_score}/10")
    
    return {
        **state,
        "confidence_score": confidence_score
    }

print("✅ Nodo EVALUATE definido!")

---

## 🎛️ Paso 10: Definir las Funciones de Decisión (Routers)

Estas funciones deciden el flujo del grafo basándose en scores y estados.

In [ ]:
def decide_after_relevance_check(state: RAGState) -> str:
    """
    Decide si continuar con la generación o buscar de nuevo.
    """
    relevance = state["relevance_score"]
    attempts = state["retrieval_attempts"]
    
    print("\n🎛️ DECISIÓN después de check de relevancia:")
    
    # Si ya intentó 3 veces, generar con lo que hay
    if attempts >= 3:
        print("   ⚠️ Máximo de intentos alcanzado → GENERATE")
        return "generate"
    
    # Si relevancia es baja, buscar de nuevo
    if relevance < 6:
        print(f"   ⚠️ Relevancia baja ({relevance}/10) → RETRIEVE de nuevo")
        return "retrieve"
    
    # Si relevancia es buena, generar respuesta
    print(f"   ✅ Relevancia buena ({relevance}/10) → GENERATE")
    return "generate"


def decide_after_evaluation(state: RAGState) -> str:
    """
    Decide si la respuesta es suficientemente buena o necesita mejora.
    """
    confidence = state["confidence_score"]
    attempts = state["retrieval_attempts"]
    
    print("\n🎛️ DECISIÓN después de evaluación:")
    
    # Si ya intentó 3 veces, terminar
    if attempts >= 3:
        print("   ⚠️ Máximo de intentos alcanzado → END")
        return END
    
    # Si confianza es baja, buscar mejores documentos
    if confidence < 7:
        print(f"   ⚠️ Confianza baja ({confidence}/10) → RETRIEVE de nuevo")
        return "retrieve"
    
    # Si confianza es alta, terminar
    print(f"   ✅ Confianza alta ({confidence}/10) → END")
    return END


print("✅ Funciones de decisión definidas!")
print("\n🎛️ Lógica de decisión:")
print("   Después de check de relevancia:")
print("      • Relevancia < 6 → Buscar de nuevo")
print("      • Relevancia ≥ 6 → Generar respuesta")
print("   Después de evaluación:")
print("      • Confianza < 7 → Buscar mejores docs")
print("      • Confianza ≥ 7 → Terminar (respuesta buena)")
print("   Límite: Máximo 3 intentos de búsqueda")

---

## 🏗️ Paso 11: Construir el Grafo Completo

Ahora conectamos todos los nodos y edges para crear el flujo completo.

In [ ]:
# Crear el grafo
workflow = StateGraph(RAGState)

# Agregar nodos
workflow.add_node("retrieve", retrieve_documents)
workflow.add_node("check_relevance", check_relevance)
workflow.add_node("generate", generate_answer)
workflow.add_node("evaluate", evaluate_answer)

# Definir entry point
workflow.set_entry_point("retrieve")

# Edges normales
workflow.add_edge("retrieve", "check_relevance")
workflow.add_edge("generate", "evaluate")

# Edges condicionales
workflow.add_conditional_edges(
    "check_relevance",
    decide_after_relevance_check,
    {
        "retrieve": "retrieve",
        "generate": "generate"
    }
)

workflow.add_conditional_edges(
    "evaluate",
    decide_after_evaluation,
    {
        "retrieve": "retrieve",
        END: END
    }
)

# Compilar
app = workflow.compile()

print("✅ Grafo RAG construido y compilado!")
print("\n🏗️ Estructura:")
print("   START → retrieve → check_relevance → [decide]")
print("              ↑              ↓")
print("              |         generate → evaluate → [decide]")
print("              |              ↓")
print("              └──────────────┘ (si necesita mejorar)")
print("                             ↓")
print("                            END (si confianza alta)")

---

## 🧪 Paso 12: Probar el Sistema RAG

### Prueba 1: Pregunta sobre tema conocido (debería funcionar bien)

In [ ]:
# Prueba 1: Pregunta sobre LangGraph (tema en la base de conocimiento)
print("="*70)
print("🧪 PRUEBA 1: Pregunta sobre tema conocido")
print("="*70)

result = app.invoke({
    "question": "¿Qué es LangGraph y para qué sirve?",
    "documents": [],
    "answer": "",
    "relevance_score": 0,
    "confidence_score": 0,
    "retrieval_attempts": 0,
    "messages": []
})

print("\n" + "="*70)
print("📊 RESULTADO FINAL")
print("="*70)
print(f"\n❓ Pregunta: {result['question']}")
print(f"\n✅ Respuesta:\n{result['answer']}")
print(f"\n📈 Métricas:")
print(f"   • Relevancia final: {result['relevance_score']}/10")
print(f"   • Confianza final: {result['confidence_score']}/10")
print(f"   • Intentos de búsqueda: {result['retrieval_attempts']}")
print(f"   • Documentos usados: {len(result['documents'])}")
print("="*70)

### Prueba 2: Pregunta más específica

In [ ]:
# Prueba 2: Pregunta específica sobre RAG
print("="*70)
print("🧪 PRUEBA 2: Pregunta específica sobre RAG")
print("="*70)

result2 = app.invoke({
    "question": "¿Cómo funciona RAG y qué beneficios tiene?",
    "documents": [],
    "answer": "",
    "relevance_score": 0,
    "confidence_score": 0,
    "retrieval_attempts": 0,
    "messages": []
})

print("\n" + "="*70)
print("📊 RESULTADO FINAL")
print("="*70)
print(f"\n❓ Pregunta: {result2['question']}")
print(f"\n✅ Respuesta:\n{result2['answer']}")
print(f"\n📈 Métricas:")
print(f"   • Relevancia final: {result2['relevance_score']}/10")
print(f"   • Confianza final: {result2['confidence_score']}/10")
print(f"   • Intentos de búsqueda: {result2['retrieval_attempts']}")
print("="*70)

### Prueba 3: Pregunta fuera del dominio (para ver self-correction)

In [ ]:
# Prueba 3: Pregunta parcialmente fuera del dominio
print("="*70)
print("🧪 PRUEBA 3: Pregunta que podría requerir múltiples intentos")
print("="*70)

result3 = app.invoke({
    "question": "¿Cuáles son las diferencias entre embeddings y vectorstores?",
    "documents": [],
    "answer": "",
    "relevance_score": 0,
    "confidence_score": 0,
    "retrieval_attempts": 0,
    "messages": []
})

print("\n" + "="*70)
print("📊 RESULTADO FINAL")
print("="*70)
print(f"\n❓ Pregunta: {result3['question']}")
print(f"\n✅ Respuesta:\n{result3['answer']}")
print(f"\n📈 Métricas:")
print(f"   • Relevancia final: {result3['relevance_score']}/10")
print(f"   • Confianza final: {result3['confidence_score']}/10")
print(f"   • Intentos de búsqueda: {result3['retrieval_attempts']}")
print("="*70)

---

## 🎯 Resumen del Proyecto

### ✅ Lo que construimos:

Un **Sistema RAG Avanzado con Self-Correction** que:

1. 🔍 **Busca documentos relevantes** usando embeddings y FAISS
2. ✅ **Evalúa la relevancia** de los documentos encontrados
3. 📝 **Genera respuestas** basadas en el contexto
4. 🎯 **Evalúa su propia confianza** en la respuesta
5. 🔄 **Se auto-corrige** buscando mejores documentos si es necesario
6. 🛡️ **Tiene límites de seguridad** (máximo 3 intentos)

---

### 🧠 Conceptos avanzados aplicados:

#### 1. **RAG Multi-iterativo**
   - No se conforma con la primera búsqueda
   - Puede intentar hasta 3 veces si no está satisfecho

#### 2. **Self-Evaluation**
   - El sistema evalúa su propia relevancia y confianza
   - Usa el LLM como juez de calidad

#### 3. **Adaptive Retrieval**
   - Ajusta su estrategia de búsqueda basándose en scores
   - Sabe cuándo necesita más información

#### 4. **Vector Search**
   - Usa embeddings para búsqueda semántica
   - FAISS para búsquedas eficientes

#### 5. **Complex Routing**
   - Múltiples puntos de decisión
   - Lógica condicional basada en scores

---

### 🆚 Comparación: RAG Básico vs RAG Avanzado

| Aspecto | RAG Básico | RAG con Self-Correction |
|---------|------------|-------------------------|
| Búsquedas | 1 sola vez | Hasta 3 intentos |
| Evaluación | Ninguna | Doble (relevancia + confianza) |
| Mejora | No | Sí, iterativa |
| Confianza | Desconocida | Medida (0-10) |
| Calidad | Variable | Más consistente |
| Complejidad | Baja | Alta |
| Nivel | Junior | Senior |

---

### 🔄 Flujo completo del sistema:

```
Pregunta: "¿Qué es LangGraph?"
    ↓
[RETRIEVE] → Busca 3 docs en vector store
    ↓
[CHECK_RELEVANCE] → Evalúa: 8/10 (Bueno!)
    ↓
[GENERATE] → Crea respuesta con contexto
    ↓
[EVALUATE] → Confianza: 9/10 (Excelente!)
    ↓
END → Respuesta de alta calidad
```

**Escenario con self-correction:**
```
Pregunta: "¿Cuál es el precio de Bitcoin?"
    ↓
[RETRIEVE] → Busca docs
    ↓
[CHECK_RELEVANCE] → 3/10 (Bajo!)
    ↓
[RETRIEVE] → Busca de nuevo (intento #2)
    ↓
[CHECK_RELEVANCE] → 4/10 (Aún bajo)
    ↓
[RETRIEVE] → Último intento (#3)
    ↓
[CHECK_RELEVANCE] → 5/10 (Aceptable)
    ↓
[GENERATE] → "No tengo información suficiente..."
    ↓
[EVALUATE] → 6/10 (Aceptable, pero honesto)
    ↓
END → Respuesta que admite limitaciones
```

---

### 💡 Mejoras posibles (para seguir aprendiendo):

1. **Web Search Fallback**
   ```python
   if relevance < 4:
       # Buscar en internet como backup
       web_results = search_web(question)
   ```

2. **Reranking**
   ```python
   # Reordenar docs por relevancia específica a la pregunta
   docs = rerank(docs, question)
   ```

3. **Query Rewriting**
   ```python
   # Reformular la pregunta para mejor búsqueda
   improved_query = rewrite_query(question, failed_attempts)
   ```

4. **Hybrid Search**
   ```python
   # Combinar búsqueda semántica + keyword
   results = hybrid_search(semantic_results, keyword_results)
   ```

5. **Confidence Calibration**
   ```python
   # Ajustar scores basándose en historial
   calibrated_score = calibrate(raw_score, historical_data)
   ```

6. **Multi-hop Reasoning**
   ```python
   # Para preguntas que requieren múltiples pasos
   sub_questions = decompose(complex_question)
   answers = [answer(q) for q in sub_questions]
   final = synthesize(answers)
   ```

---

### 🎓 ¿Por qué esto es nivel Senior?

1. **Pensamiento sistémico**: No es solo "hacer RAG", es diseñar un sistema que se mejora a sí mismo
2. **Manejo de incertidumbre**: Sabe cuándo no sabe y busca mejorar
3. **Múltiples capas de decisión**: No hay un solo "if-else"
4. **Evaluación automática**: El sistema se califica a sí mismo
5. **Graceful degradation**: Si falla 3 veces, no rompe, responde honestamente
6. **Production-ready patterns**: Límites, timeouts, fallbacks

---

### 🚀 Casos de uso reales:

Este patrón se usa en:
- 📚 **Asistentes de documentación técnica**
- 🏥 **Sistemas médicos de soporte a diagnóstico**
- ⚖️ **Legal research assistants**
- 🔬 **Scientific literature review**
- 💼 **Enterprise knowledge bases**
- 🎓 **Educational tutoring systems**

---

### 🏆 ¡Felicidades!

Has construido un sistema RAG de nivel profesional. Este tipo de arquitectura es lo que separa:

- ❌ Un "RAG básico de tutorial" 
- ✅ Un **sistema de producción confiable**

Este proyecto demuestra:
- ✅ Comprensión profunda de RAG
- ✅ Capacidad de diseñar sistemas complejos
- ✅ Pensamiento en calidad y confiabilidad
- ✅ Habilidades de nivel senior

**¡Excelente trabajo! 🎉**

---

## 💪 Ejercicios de Extensión

### Ejercicio 1: Agregar más documentos

Agrega 5 documentos nuevos sobre temas de tu elección y prueba el sistema.

### Ejercicio 2: Ajustar umbrales

Modifica los umbrales de relevancia y confianza:
- Cambiar de 6 a 7 el umbral de relevancia
- Cambiar de 7 a 8 el umbral de confianza
- Observar cómo afecta el comportamiento

### Ejercicio 3: Agregar logging detallado

Crea un sistema de logging que guarde:
- Todas las preguntas
- Documentos recuperados en cada intento
- Scores de cada evaluación
- Tiempo de respuesta

### Ejercicio 4: Implementar web search fallback

Si después de 3 intentos la relevancia sigue baja, buscar en internet.

### Ejercicio 5: Crear una interfaz Streamlit

Construye una interfaz web simple donde:
- Usuario ingresa pregunta
- Muestra el proceso en tiempo real
- Visualiza los scores
- Muestra documentos usados

---

¡Experimenta y lleva este sistema al siguiente nivel! 🚀